# CAND-VB2 RAG — Google Colab

RAG chuyên biệt về **VB2CA tuyển mới** dành cho công dân đã có bằng đại học, ưu tiên trường hợp văn bằng 1 CNTT/IT.

## Cách dùng

- **Lần đầu / khi dataset hoặc quy định được cập nhật:** chạy cell `FULL UPDATE / BUILD` bên dưới. Index được lưu bền vững tại `MyDrive/CAND_VB2_RAG/index`.
- **Những lần chat sau:** chỉ cần chạy **CELL CUỐI — QUICK RESUME & CHAT**. Cell cuối tự mount Drive, clone/update code, đọc `/content/providers.env`, nạp index cũ và mở Gradio. **Không crawl và không embedding lại.**
- Gemini + OpenRouter là API model nên không có weight/checkpoint local để lưu. Phần được persist là knowledge index + metadata/version manifest.
- `providers.env` không bao giờ được commit lên GitHub. Ở runtime mới, nếu `/content/providers.env` chưa có, cell sẽ yêu cầu bạn upload.


## FULL UPDATE / BUILD — chỉ chạy khi dữ liệu thay đổi

Cell này xác minh cấu trúc nguồn, build index từ `data/corpus/` và lưu sang Google Drive. Nếu fingerprint corpus không đổi, script tự bỏ qua re-embedding.


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess
from google.colab import drive, files

REPO_URL = 'https://github.com/NVTruong473/NLP.git'
REPO_DIR = Path('/content/NLP')
ENV_PATH = Path('/content/providers.env')
DRIVE_ROOT = Path('/content/drive/MyDrive/CAND_VB2_RAG')
INDEX_DIR = DRIVE_ROOT / 'index'

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)

if not ENV_PATH.exists():
    print('Upload providers.env -> /content/providers.env')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('Thiếu providers.env')
    selected = next(iter(uploaded))
    Path(selected).replace(ENV_PATH)

os.environ['VIETRAG_INDEX_DIR'] = str(INDEX_DIR)
os.environ['VIETRAG_ENV'] = str(ENV_PATH)

subprocess.run([sys.executable, 'scripts/validate_sources.py'], check=True)
subprocess.run([
    sys.executable, 'scripts/build_index.py',
    '--input-dir', 'data/corpus',
    '--env', str(ENV_PATH),
], check=True)

manifest = INDEX_DIR / 'index_manifest.json'
print('\nFULL BUILD READY')
print('Persistent index:', INDEX_DIR)
print('Manifest:', manifest)
print('Từ lần sau, chỉ chạy CELL CUỐI để chat.')


## CELL CUỐI — QUICK RESUME & CHAT

**Đây là cell duy nhất cần chạy ở những lần sử dụng sau.** Cell này không gọi `build_index.py`, không embedding lại và launch Gradio trực tiếp trong kernel Colab. Nó chủ động thêm `src/` vào `sys.path`, nên không phụ thuộc việc editable install đã được kernel hiện tại nhận hay chưa.


In [ ]:
# QUICK RESUME: fresh Colab runtime -> one cell -> chat
from pathlib import Path
import os, sys, subprocess, importlib, importlib.util, json
from google.colab import drive, files

REPO_URL = 'https://github.com/NVTruong473/NLP.git'
REPO_DIR = Path('/content/NLP')
SRC_DIR = REPO_DIR / 'src'
ENV_PATH = Path('/content/providers.env')
INDEX_DIR = Path('/content/drive/MyDrive/CAND_VB2_RAG/index')

def step(message):
    print(f'[CAND-VB2] {message}', flush=True)

# 1) Persistent knowledge index
step('Mounting Google Drive...')
drive.mount('/content/drive', force_remount=False)
required_index = [INDEX_DIR/'faiss.index', INDEX_DIR/'chunks.jsonl', INDEX_DIR/'index_manifest.json']
if not all(p.exists() for p in required_index):
    missing = [str(p) for p in required_index if not p.exists()]
    raise RuntimeError('Chưa có persistent index. Hãy chạy FULL UPDATE / BUILD một lần. Missing: ' + ', '.join(missing))

# 2) Current source code — no data rebuild
step('Syncing current repo code (no rebuild)...')
if (REPO_DIR/'.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

# Critical for the CURRENT Colab kernel: editable pip installs run in a subprocess,
# so their .pth changes are not guaranteed to appear in this already-running interpreter.
# Add the src-layout explicitly before importing app/vietrag.
src_path = str(SRC_DIR)
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()
if importlib.util.find_spec('vietrag') is None:
    raise RuntimeError(f'Không import được vietrag dù đã thêm {src_path} vào sys.path')
step(f'Python package path ready: {src_path}')

# Install only when a fresh runtime is missing third-party modules.
required_modules = ['faiss', 'gradio', 'rank_bm25', 'dotenv', 'google.genai', 'yaml', 'requests']
missing_modules = []
for name in required_modules:
    try:
        if importlib.util.find_spec(name) is None:
            missing_modules.append(name)
    except (ModuleNotFoundError, ValueError):
        missing_modules.append(name)
if missing_modules:
    step('Installing runtime dependencies: ' + ', '.join(missing_modules))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    importlib.invalidate_caches()

# 3) Private API keys stay only in /content for this runtime.
if not ENV_PATH.exists():
    step('Không thấy /content/providers.env — hãy upload file providers.env.')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('Thiếu providers.env')
    selected = next(iter(uploaded))
    Path(selected).replace(ENV_PATH)

# 4) Point the app to the SAVED index. No crawling, embedding or rebuilding.
os.environ['VIETRAG_INDEX_DIR'] = str(INDEX_DIR)
os.environ['VIETRAG_ENV'] = str(ENV_PATH)
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'

manifest = json.loads((INDEX_DIR/'index_manifest.json').read_text(encoding='utf-8'))
step(f'Loaded saved index: {INDEX_DIR}')
print('Built at:', manifest.get('created_at_utc'), flush=True)
print('Chunks:', manifest.get('chunk_count'), '| Sources:', len(manifest.get('source_ids', [])), flush=True)
print('Fingerprint:', str(manifest.get('fingerprint', ''))[:16], flush=True)

# 5) Load app.py directly in this Colab kernel so Gradio output is visible.
step('Loading RAG pipeline from saved index...')
for module_name in list(sys.modules):
    if module_name == 'app' or module_name == 'vietrag' or module_name.startswith('vietrag.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
import app
step('Pipeline ready. Launching Gradio...')
launch_result = app.demo.launch(
    share=True,
    inline=True,
    prevent_thread_lock=True,
    show_error=True,
)
try:
    _, local_url, public_url = launch_result
except Exception:
    local_url = getattr(app.demo, 'local_url', None)
    public_url = getattr(app.demo, 'share_url', None)
print('\n=== CHAT READY ===', flush=True)
print('Local URL :', local_url, flush=True)
print('Public URL:', public_url or 'Không tạo được public tunnel; dùng giao diện inline phía trên.', flush=True)
print('NO RE-EMBEDDING. Dataset/index cũ vẫn được dùng.', flush=True)
